# 📊 Analyse et Validation des Données (ETL)

Ce notebook regroupe l'exploration des données brutes (Collecte), des données nettoyées (Transformation) et le chargement en base de données (Load).

### Pipeline ETL :
1. **Collecte (`data/raw`)** : Récupération depuis API et Scraping.
2. **Transformation (`data/processed`)** : Nettoyage et standardisation.
3. **Chargement (PostgreSQL)** : Insertion des données structurées en base.
---

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Ajout du dossier racine au path pour les imports de modules src
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Configuration
RAW_DIR = Path("../data/raw").resolve()
PROCESSED_DIR = Path("../data/processed").resolve()

print(f"📂 RAW_DIR: {RAW_DIR}")
print(f"📂 PROCESSED_DIR: {PROCESSED_DIR}")

# PARTIE 1 : Données Brutes (Collecte)

In [ ]:
# 1. Légifrance (Brut)
file_legifrance = RAW_DIR / "legifrance.csv"
if file_legifrance.exists():
    df = pd.read_csv(file_legifrance)
    print(f"📜 Légifrance (Raw): {len(df)} articles")
    display(df.head(2))

# 2. INRS (Brut)
file_inrs = RAW_DIR / "inrs.csv"
if file_inrs.exists():
    df = pd.read_csv(file_inrs)
    print(f"📚 INRS (Raw): {len(df)} guides")
    display(df.head(2))

# 3. ARIA (Brut)
file_aria = RAW_DIR / "aria_accidents.csv"
if file_aria.exists():
    try:
        df = pd.read_csv(file_aria, sep=";", encoding="latin-1", on_bad_lines='skip')
        if len(df.columns) <= 1:
             df = pd.read_csv(file_aria, sep=",", encoding="latin-1", on_bad_lines='skip')
        print(f"🏭 ARIA (Raw): {len(df)} lignes")
        display(df.head(2))
    except Exception as e: print(e)

# 4. WAQI (Brut)
file_waqi = RAW_DIR / "waqi.csv"
if file_waqi.exists():
    df = pd.read_csv(file_waqi)
    print(f"💨 WAQI (Raw): {len(df)} relevés")
    display(df.head(2))

# PARTIE 2 : Données Transformées (Nettoyage)
Vérification des fichiers dans `data/processed/`.

## 2.1 Qualité de l'Air (WAQI Cleaned)

In [ ]:
file_waqi_clean = PROCESSED_DIR / "waqi_cleaned.csv"

if file_waqi_clean.exists():
    df_waqi = pd.read_csv(file_waqi_clean)
    print(f"✅ Chargé : {len(df_waqi)} lignes")
    
    # Aperçu avec les nouvelles colonnes
    cols_to_show = ['ville_recherchee', 'aqi', 'niveau_risque', 'conseil_qhse']
    display(df_waqi[cols_to_show])
    
    # Visualisation simple AQI
    if not df_waqi.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=df_waqi, x='ville_recherchee', y='aqi', palette='viridis')
        plt.title("Qualité de l'Air (AQI) par Ville")
        plt.axhline(50, color='g', linestyle='--', label='Bon (50)')
        plt.axhline(100, color='orange', linestyle='--', label='Modéré (100)')
        plt.legend()
        plt.show()
else:
    print("❌ Fichier waqi_cleaned.csv manquant")

## 2.2 Accidents Industriels (ARIA Cleaned)

In [ ]:
file_aria_clean = PROCESSED_DIR / "aria_cleaned.csv"

if file_aria_clean.exists():
    df_aria = pd.read_csv(file_aria_clean)
    print(f"✅ Chargé : {len(df_aria)} accidents")
    
    # Vérification colonnes normalisées
    print("Colonnes :", df_aria.columns.tolist())
    display(df_aria.head(3))
else:
    print("❌ Fichier aria_cleaned.csv manquant")

## 2.3 Réglementation & Guides (Légifrance & INRS Cleaned)

In [ ]:
file_legi_clean = PROCESSED_DIR / "legifrance_cleaned.csv"
file_inrs_clean = PROCESSED_DIR / "inrs_cleaned.csv"

if file_legi_clean.exists():
    df_legi = pd.read_csv(file_legi_clean)
    print(f"📜 Légifrance : {len(df_legi)} articles prêts")
    print(f"   Exemple titre: {df_legi.iloc[0]['titre']}")

if file_inrs_clean.exists():
    df_inrs = pd.read_csv(file_inrs_clean)
    print(f"📚 INRS : {len(df_inrs)} guides prêts")

# PARTIE 3 : Chargement en Base de Données (Load)
Exécution du module `src.etl.load` directement depuis le notebook pour peupler PostgreSQL.

In [ ]:
from src.etl.load import DataLoader
from src.monitoring.logger import logger
import logging

# Afficher les logs dans la sortie du notebook
logging.getLogger().setLevel(logging.INFO)

try:
    print("🚀 Initialisation du DataLoader...")
    loader = DataLoader()
    
    # 3.1 Chargement Légifrance
    loader.load_legifrance()
    
    # 3.2 Chargement INRS
    loader.load_inrs()
    
    # 3.3 Chargement WAQI
    loader.load_waqi()
    
    # 3.4 Chargement ARIA
    loader.load_aria()
    
    loader.close()
    print("🎉 Chargement terminé avec succès !")
    
except Exception as e:
    print(f"❌ Erreur lors du chargement : {e}")